# 09 — Multi Output Pattern

One pipeline → multiple outputs (silver, quarantine, metrics)

Process:

INPUT
  |
  v
PREPARE
  |
  +--> SILVER
  +--> QUARANTINE
  +--> METRICS

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("multi-output-pattern")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Input

In [2]:
df = spark.createDataFrame(
    [
        ("e1","u1",10.0),
        ("e2",None,20.0),
        ("e3","u2",-5.0),
        ("e4","u3",15.0),
    ],
    ["event_id","user_id","amount"]
)

df.show()

+--------+-------+------+
|event_id|user_id|amount|
+--------+-------+------+
|      e1|     u1|  10.0|
|      e2|   NULL|  20.0|
|      e3|     u2|  -5.0|
|      e4|     u3|  15.0|
+--------+-------+------+



## Step 2 — Prepare

In [3]:
prepared = (
    df
    .withColumn("error_reason",
        F.when(F.col("user_id").isNull(), F.lit("missing_user"))
         .when(F.col("amount") < 0, F.lit("negative_amount"))
    )
    .withColumn("is_valid", F.col("error_reason").isNull())
)

## Step 3 — Outputs

In [4]:
silver = prepared.filter("is_valid")
quarantine = prepared.filter("NOT is_valid")

metrics = spark.createDataFrame(
    [
        ("input_rows", df.count()),
        ("valid_rows", silver.count()),
        ("invalid_rows", quarantine.count()),
    ],
    ["metric","value"]
)

silver.show()
quarantine.show()
metrics.show()

+--------+-------+------+------------+--------+
|event_id|user_id|amount|error_reason|is_valid|
+--------+-------+------+------------+--------+
|      e1|     u1|  10.0|        NULL|    true|
|      e4|     u3|  15.0|        NULL|    true|
+--------+-------+------+------------+--------+

+--------+-------+------+---------------+--------+
|event_id|user_id|amount|   error_reason|is_valid|
+--------+-------+------+---------------+--------+
|      e2|   NULL|  20.0|   missing_user|   false|
|      e3|     u2|  -5.0|negative_amount|   false|
+--------+-------+------+---------------+--------+

+------------+-----+
|      metric|value|
+------------+-----+
|  input_rows|    4|
|  valid_rows|    2|
|invalid_rows|    2|
+------------+-----+



## Step 4 — Return pattern

In [5]:
def run_pipeline(df):
    prepared = (
        df
        .withColumn("error_reason",
            F.when(F.col("user_id").isNull(), F.lit("missing_user"))
             .when(F.col("amount") < 0, F.lit("negative_amount"))
        )
        .withColumn("is_valid", F.col("error_reason").isNull())
    )

    return {
        "silver": prepared.filter("is_valid"),
        "quarantine": prepared.filter("NOT is_valid"),
        "metrics": spark.createDataFrame(
            [
                ("input", df.count()),
                ("valid", prepared.filter("is_valid").count()),
                ("invalid", prepared.filter("NOT is_valid").count()),
            ],
            ["metric","value"]
        )
    }

result = run_pipeline(df)

result["metrics"].show()

+-------+-----+
| metric|value|
+-------+-----+
|  input|    4|
|  valid|    2|
|invalid|    2|
+-------+-----+



In [6]:
spark.stop()